# quantbench_paper_soxl_v2

Executable evidence for the expanded QuantBench-to-SOXL architecture matrix after implementing the missing paper model families in `src`.

This notebook runs repo-owned app-layer logic only: model registry/fit lives in `src.soxl_quant.app.quantbench_supervised`, coverage in `src.soxl_quant.app.quantbench_screen`, and path ensembles in `src.soxl_quant.app.quantbench_ensemble`.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
CARD = ROOT / "runtime" / "model_selection" / "card" / "quantbench_paper_soxl_v2"
CARD.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import torch
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass

from scripts.update_market_data import _fetch_cboe_vix, _fetch_yahoo_api
from src.soxl_quant.app.futu import check_opend_port
from src.soxl_quant.app.quantbench_screen import QuantBenchGate, build_quantbench_screen_report
from src.soxl_quant.app.quantbench_supervised import (
    build_quantbench_supervised_report,
    quantbench_supervised_model_specs,
)
from src.soxl_quant.app.quantbench_ensemble import build_quantbench_supervised_ensemble_report
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

print("root=.")
print(f"card={CARD.relative_to(ROOT)}")

root=.
card=runtime/model_selection/card/quantbench_paper_soxl_v2


## Data Freshness

The canonical daily panel is the source for the model backtest. Live refresh probes are recorded separately so the run does not mix manual web OHLCV into the canonical dataset.

In [2]:
def csv_date_range(relative_path: str, date_candidates=("date", "DATE", "time_key")) -> dict[str, object]:
    path = ROOT / relative_path
    if not path.exists():
        return {"path": relative_path, "exists": False}
    frame = pd.read_csv(path)
    date_col = next((column for column in date_candidates if column in frame.columns), frame.columns[0])
    dates = pd.to_datetime(frame[date_col], errors="coerce")
    return {
        "path": relative_path,
        "exists": True,
        "rows": int(len(frame)),
        "start": None if dates.dropna().empty else str(dates.min().date()),
        "end": None if dates.dropna().empty else str(dates.max().date()),
        "date_col": str(date_col),
    }

data_surfaces = {
    "canonical_daily": csv_date_range("data/processed/panels/soxl_soxx_vix_full.csv"),
    "raw_soxl": csv_date_range("data/raw/symbols/SOXL.csv"),
    "raw_soxx": csv_date_range("data/raw/symbols/SOXX.csv"),
    "raw_vix": csv_date_range("data/raw/external/VIX.csv", date_candidates=("DATE", "date")),
    "intraday_daily_safe": csv_date_range("data/raw/external/intraday_features_soxl_1m.csv"),
    "futu_daily_soxl": csv_date_range("data/raw/external/futu_history_kline_US_SOXL.csv"),
    "tushare_daily_soxl": csv_date_range("data/raw/external/tushare_us_daily_adj_SOXL.csv"),
}

live_refresh_probe: dict[str, object] = {"yahoo_api": {}, "cboe_vix": {}, "futu_opend_reachable": False}
for ticker in ("SOXL", "SOXX", "^VIX"):
    try:
        frame = _fetch_yahoo_api(ticker, start="2026-05-25")
        live_refresh_probe["yahoo_api"][ticker] = {
            "status": "ok",
            "rows": int(len(frame)),
            "end": str(pd.to_datetime(frame["date"]).max().date()),
        }
    except Exception as exc:
        live_refresh_probe["yahoo_api"][ticker] = {"status": "failed", "error": str(exc)[:260]}
try:
    vix_frame = _fetch_cboe_vix(start="2026-05-25")
    live_refresh_probe["cboe_vix"] = {
        "status": "ok",
        "rows": int(len(vix_frame)),
        "end": str(pd.to_datetime(vix_frame["date"]).max().date()),
        "latest_vix": float(vix_frame.sort_values("date").iloc[-1]["vix"]),
    }
except Exception as exc:
    live_refresh_probe["cboe_vix"] = {"status": "failed", "error": str(exc)[:260]}
live_refresh_probe["futu_opend_reachable"] = bool(check_opend_port(timeout=1.0))

data_refresh_status = {
    "data_surfaces": data_surfaces,
    "live_refresh_probe": live_refresh_probe,
    "canonical_daily_end": data_surfaces["canonical_daily"].get("end"),
    "model_backtest_data_end": data_surfaces["canonical_daily"].get("end"),
    "note": "Yahoo API was probed live; model backtest uses canonical local OHLCV only.",
}
(CARD / "data_refresh_status.json").write_text(json.dumps(data_refresh_status, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(data_refresh_status, indent=2, sort_keys=True))

{
  "canonical_daily_end": "2026-06-04",
  "data_surfaces": {
    "canonical_daily": {
      "date_col": "date",
      "end": "2026-06-04",
      "exists": true,
      "path": "data/processed/panels/soxl_soxx_vix_full.csv",
      "rows": 4084,
      "start": "2010-03-11"
    },
    "futu_daily_soxl": {
      "date_col": "date",
      "end": "2026-05-22",
      "exists": true,
      "path": "data/raw/external/futu_history_kline_US_SOXL.csv",
      "rows": 4060,
      "start": "2010-03-31"
    },
    "intraday_daily_safe": {
      "date_col": "date",
      "end": "2026-06-05",
      "exists": true,
      "path": "data/raw/external/intraday_features_soxl_1m.csv",
      "rows": 608,
      "start": "2024-01-03"
    },
    "raw_soxl": {
      "date_col": "date",
      "end": "2026-06-04",
      "exists": true,
      "path": "data/raw/symbols/SOXL.csv",
      "rows": 4084,
      "start": "2010-03-11"
    },
    "raw_soxx": {
      "date_col": "date",
      "end": "2026-06-04",
      "exists":

## Expanded Architecture Matrix

The full supervised registry now includes the previously missing tree, sequence, attention, transformer, graph, residual, and hypergraph families.

In [3]:
model_specs = quantbench_supervised_model_specs()
expanded_model_names = tuple(spec.name for spec in model_specs)
model_specs_df = pd.DataFrame([spec.__dict__ for spec in model_specs])
print({"model_count": len(expanded_model_names), "model_names": list(expanded_model_names)})
display(model_specs_df)

{'model_count': 28, 'model_names': ['logistic_clf', 'ridge_ic', 'ridge_mse', 'random_forest_clf', 'hist_gradient_mse', 'xgboost_mse', 'lightgbm_mse', 'catboost_rank', 'extra_trees_rank', 'mlp_mse', 'sequence_lag_ridge_mse', 'lstm_mse', 'gru_mse', 'alstm_mse', 'sfm_mse', 'darnn_mse', 'hawkes_gru_mse', 'tcn_mse', 'mlp_mixer_mse', 'dlinear_mse', 'patch_tst_mse', 'autoformer_mse', 'market_graph_mse', 'gcn_mse', 'gat_mse', 'rgcn_mse', 'rsr_mse', 'hypergraph_mse']}


,name,paper_family,objective,target_kind,estimator
0,logistic_clf,linear_classifier,clf,binary,LogisticRegression
1,ridge_ic,linear_regression,ic,return,Ridge
2,ridge_mse,linear_regression,mse,return,Ridge
3,random_forest_clf,tree,clf,binary,RandomForestClassifier
4,hist_gradient_mse,boosted_tree,mse,return,HistGradientBoostingRegressor
5,xgboost_mse,boosted_tree_xgboost,mse,return,XGBoostRegressorOrHistGradientFallback
6,lightgbm_mse,boosted_tree_lightgbm,mse,return,LightGBMRegressorOrHistGradientFallback
7,catboost_rank,boosted_tree_catboost,ranking,return,CatBoostRegressorOrExtraTreesFallback
8,extra_trees_rank,tree_ensemble,ranking,return,ExtraTreesRegressor
9,mlp_mse,neural_tabular,mse,return,MLPRegressor


## Execute Src-Owned QuantBench Screen

In [4]:
gate = QuantBenchGate(
    max_drawdown_pct=20.0,
    min_stress_relative_return=30.0,
    min_stress_avg_pos=0.10,
    max_stress_avg_pos=0.45,
    min_bull_capture=0.50,
    min_proxy_return=0.0,
)
report = build_quantbench_screen_report(root=ROOT, card_dir=CARD, gate=gate)
print(json.dumps({
    "decision": report["decision"],
    "records": report["screen"]["records"],
    "eligible_profiles": report["screen"]["eligible_profiles"],
}, indent=2))

{
  "decision": "no_current_model_satisfies_quantbench_soxl_20dd_gate",
  "records": 26,
  "eligible_profiles": []
}


## Build Dataset And Windows

In [5]:
feature_packs = (
    "vix_term",
    "academic_signals",
    "macro_signals",
    "rulebase_signals",
    "transition_signals",
    "soxx_signals",
    "soxx_spread_oscillation",
)

def index_dates(index: pd.Index) -> pd.DatetimeIndex:
    if isinstance(index, pd.MultiIndex):
        level = "datetime" if "datetime" in index.names else 0
        return pd.DatetimeIndex(pd.to_datetime(index.get_level_values(level)))
    return pd.DatetimeIndex(pd.to_datetime(index))

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=feature_packs, profile_name="quantbench_paper_soxl_v2")
features = dataset.handler.feature_frame
close = dataset.handler.price_frame["price"]
feature_dates = index_dates(features.index)
close_dates = index_dates(close.index)
feature_end = str(feature_dates.max().date())
close_end = str(close_dates.max().date())
model_test_end = min(feature_end, close_end)
windows = {
    "test_2020": ("2020-01-01", "2020-12-31"),
    "test_2022": ("2022-01-01", "2022-12-31"),
    "test_2023_2024": ("2023-01-01", "2024-12-31"),
    "latest_2025_2026": ("2025-01-01", model_test_end),
    "ytd_2026": ("2026-01-01", model_test_end),
}
print(json.dumps({
    "feature_rows": int(len(features)),
    "feature_count": int(features.shape[1]),
    "feature_start": str(feature_dates.min().date()),
    "feature_end": feature_end,
    "close_end": close_end,
    "model_test_end": model_test_end,
    "windows": windows,
}, indent=2))

[88215:MainThread](2026-06-07 11:35:41,546) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[88215:MainThread](2026-06-07 11:35:42,002) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[88215:MainThread](2026-06-07 11:35:42,002) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/.worktrees/quantbench-full-architectures-v1/data/qlib/soxl')}


[88215:MainThread](2026-06-07 11:35:43,581) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[88215:MainThread](2026-06-07 11:35:43,582) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[88215:MainThread](2026-06-07 11:35:43,582) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{
  "feature_rows": 3831,
  "feature_count": 113,
  "feature_start": "2011-03-11",
  "feature_end": "2026-06-04",
  "close_end": "2026-06-04",
  "model_test_end": "2026-06-04",
  "windows": {
    "test_2020": [
      "2020-01-01",
      "2020-12-31"
    ],
    "test_2022": [
      "2022-01-01",
      "2022-12-31"
    ],
    "test_2023_2024": [
      "2023-01-01",
      "2024-12-31"
    ],
    "latest_2025_2026": [
      "2025-01-01",
      "2026-06-04"
    ],
    "ytd_2026": [
      "2026-01-01",
      "2026-06-04"
    ]
  }
}


## Execute Full Supervised Model Matrix

In [6]:
supervised_report = build_quantbench_supervised_report(
    features,
    close,
    card_dir=CARD,
    root=ROOT,
    horizon=20,
    model_names=expanded_model_names,
    feature_packs=feature_packs,
    gate=gate,
    windows=windows,
    test_end=model_test_end,
    rolling_model_names=("logistic_clf", "sequence_lag_ridge_mse", "gru_mse"),
)
print(json.dumps({
    "decision": supervised_report["decision"],
    "records": supervised_report["screen"]["records"],
    "eligible_profiles": supervised_report["screen"]["eligible_profiles"],
    "feature_count": supervised_report["data"]["feature_count"],
    "model_count": len(supervised_report["data"]["model_names"]),
}, indent=2))

{
  "decision": "no_supervised_model_satisfies_quantbench_soxl_20dd_gate",
  "records": 155,
  "eligible_profiles": [],
  "feature_count": 113,
  "model_count": 31
}


## Execute Full Path Ensemble Backtest

In [7]:
ensemble_report = build_quantbench_supervised_ensemble_report(
    features,
    close,
    card_dir=CARD,
    root=ROOT,
    horizon=20,
    model_names=expanded_model_names,
    feature_packs=feature_packs,
    gate=gate,
    windows=windows,
    test_end=model_test_end,
    rolling_model_names=("logistic_clf", "sequence_lag_ridge_mse", "gru_mse"),
)
print(json.dumps({
    "decision": ensemble_report["decision"],
    "records": ensemble_report["screen"]["records"],
    "eligible_profiles": ensemble_report["screen"]["eligible_profiles"],
    "best_profile": ensemble_report["screen"]["best_rows"][0]["profile"] if ensemble_report["screen"]["best_rows"] else None,
    "model_count": len(ensemble_report["data"]["model_names"]),
}, indent=2))

{
  "decision": "eligible_ensemble_candidates_found",
  "records": 27,
  "eligible_profiles": [
    "quantbench_ensemble_sequence_fragile_panic_budget_v1"
  ],
  "best_profile": "quantbench_ensemble_sequence_fragile_panic_budget_v1",
  "model_count": 31
}


## Latest 2025-2026 Return Readout

In [8]:
def payload_latest_rows(payload_path: Path, slice_name: str = "latest_2025_2026") -> pd.DataFrame:
    payloads = json.loads(payload_path.read_text(encoding="utf-8"))
    rows = []
    for payload in payloads:
        latest = (payload.get("slices") or {}).get(slice_name) or {}
        if not latest:
            continue
        rows.append({
            "profile": payload.get("profile"),
            "model_name": payload.get("model_name"),
            "paper_family": payload.get("paper_family"),
            "objective": payload.get("objective"),
            "risk_overlay": (payload.get("risk_overlay") or {}).get("name", ""),
            "latest_return": latest.get("return"),
            "latest_buy_hold_return": latest.get("buy_and_hold_return"),
            "latest_relative_return": latest.get("relative_return"),
            "latest_max_dd": latest.get("max_dd"),
            "latest_avg_pos": latest.get("avg_pos"),
            "proxy_return": (payload.get("test_proxy") or {}).get("return"),
            "proxy_max_dd": (payload.get("test_proxy") or {}).get("max_dd"),
        })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["latest_return", "proxy_return"], ascending=False)

supervised_latest_df = payload_latest_rows(CARD / "supervised_payloads.json", "latest_2025_2026")
ensemble_latest_df = payload_latest_rows(CARD / "ensemble_payloads.json", "latest_2025_2026")
supervised_ytd_2026_df = payload_latest_rows(CARD / "supervised_payloads.json", "ytd_2026")
ensemble_ytd_2026_df = payload_latest_rows(CARD / "ensemble_payloads.json", "ytd_2026")
supervised_screen_df = pd.read_csv(CARD / "supervised_candidate_screen.csv")
ensemble_screen_df = pd.read_csv(CARD / "ensemble_candidate_screen.csv")
coverage_df = pd.read_csv(CARD / "paper_model_objective_coverage.csv")

print({
    "supervised_latest_rows": len(supervised_latest_df),
    "ensemble_latest_rows": len(ensemble_latest_df),
    "supervised_ytd_2026_rows": len(supervised_ytd_2026_df),
    "ensemble_ytd_2026_rows": len(ensemble_ytd_2026_df),
    "supervised_gate_eligible": int(supervised_screen_df["eligible"].sum()) if not supervised_screen_df.empty else 0,
    "ensemble_gate_eligible": int(ensemble_screen_df["eligible"].sum()) if not ensemble_screen_df.empty else 0,
})
display(supervised_latest_df.head(20))
display(ensemble_latest_df.head(20))
display(supervised_ytd_2026_df.head(20))
display(ensemble_ytd_2026_df.head(20))
display(ensemble_screen_df.head(20))
display(coverage_df)

{'supervised_latest_rows': 155, 'ensemble_latest_rows': 27, 'supervised_ytd_2026_rows': 155, 'ensemble_ytd_2026_rows': 27, 'supervised_gate_eligible': 0, 'ensemble_gate_eligible': 1}


,profile,model_name,paper_family,objective,risk_overlay,latest_return,latest_buy_hold_return,latest_relative_return,latest_max_dd,latest_avg_pos,proxy_return,proxy_max_dd
85,quantbench_supervised_tcn_mse,tcn_mse,sequence_convolutional,mse,,492.647537,849.403728,-356.756191,46.761492,0.505310,676.317141,71.860002
45,quantbench_supervised_mlp_mse,mlp_mse,neural_tabular,mse,,414.064805,849.403728,-435.338923,55.849663,0.478752,1498.768596,67.409087
60,quantbench_supervised_gru_mse,gru_mse,sequence_recurrent,mse,,327.910390,849.403728,-521.493338,57.361126,0.500482,719.470174,67.461955
40,quantbench_supervised_extra_trees_rank,extra_trees_rank,tree_ensemble,ranking,,299.780567,849.403728,-549.623160,60.311527,0.491586,325.009533,78.137709
25,quantbench_supervised_xgboost_mse,xgboost_mse,boosted_tree_xgboost,mse,,298.968675,849.403728,-550.435053,55.166912,0.491968,654.293118,71.210966
20,quantbench_supervised_hist_gradient_mse,hist_gradient_mse,boosted_tree,mse,,296.884896,849.403728,-552.518832,55.597366,0.491586,681.761273,70.883673
55,quantbench_supervised_lstm_mse,lstm_mse,sequence_recurrent,mse,,294.640225,849.403728,-554.763503,50.391519,0.495272,762.057810,67.489228
35,quantbench_supervised_catboost_rank,catboost_rank,boosted_tree_catboost,ranking,,288.115127,849.403728,-561.288601,62.607632,0.491078,250.233569,78.495782
65,quantbench_supervised_alstm_mse,alstm_mse,sequence_attention_recurrent,mse,,260.083549,849.403728,-589.320178,59.653080,0.481802,337.854882,70.637141
70,quantbench_supervised_sfm_mse,sfm_mse,sequence_frequency,mse,,249.673925,849.403728,-599.729803,55.854415,0.489108,348.892273,73.683676


,profile,model_name,paper_family,objective,risk_overlay,latest_return,latest_buy_hold_return,latest_relative_return,latest_max_dd,latest_avg_pos,proxy_return,proxy_max_dd
3,quantbench_ensemble_defensive_logistic_v1,None,None,None,,86.136252,849.403728,-763.267476,17.582528,0.299290,72.794366,44.705274
20,quantbench_ensemble_hazard_rf_logistic_v1,None,None,None,,85.161885,849.403728,-764.241843,29.007614,0.357212,192.180511,47.870092
4,quantbench_ensemble_guarded_rf_logistic_v1,None,None,None,,71.631096,849.403728,-777.772632,29.007614,0.382015,250.816294,47.145779
22,quantbench_ensemble_external_crash_v19v22_v1,None,None,None,,60.820697,849.403728,-788.583031,13.715434,0.194321,166.842013,17.512531
5,quantbench_ensemble_guarded_hist_logistic_v1,None,None,None,,59.533761,849.403728,-789.869967,29.392278,0.373869,247.457440,47.336439
0,quantbench_ensemble_lowdd_mlp_v1,None,None,None,,58.057493,849.403728,-791.346235,20.421334,0.286578,43.170834,45.967561
2,quantbench_ensemble_extra_logistic_v1,None,None,None,,56.049086,849.403728,-793.354642,19.841527,0.275645,28.607388,45.802637
1,quantbench_ensemble_lowdd_ridge_v1,None,None,None,,53.674973,849.403728,-795.728754,18.579882,0.269604,35.258738,44.044557
21,quantbench_ensemble_external_crash_condvol_v1,None,None,None,,53.324923,849.403728,-796.078804,13.200281,0.156576,120.426340,17.902942
13,quantbench_ensemble_sequence_full_rebase_tail_v1,None,None,None,,50.038094,849.403728,-799.365634,12.238460,0.174152,323.538434,24.011303


,profile,model_name,paper_family,objective,risk_overlay,latest_return,latest_buy_hold_return,latest_relative_return,latest_max_dd,latest_avg_pos,proxy_return,proxy_max_dd
85,quantbench_supervised_tcn_mse,tcn_mse,sequence_convolutional,mse,,219.749850,456.096534,-236.346684,13.286691,0.502391,676.317141,71.860002
150,quantbench_supervised_rolling_gru_mse,rolling_gru_mse,custom,custom,,195.797564,456.096534,-260.298970,23.033279,0.512396,526.718703,70.573906
86,quantbench_supervised_tcn_mse_macro_budget_def...,tcn_mse,sequence_convolutional,mse,macro_budget_defensive_v1,153.424904,456.096534,-302.671630,16.353040,0.437124,118.115407,24.456710
35,quantbench_supervised_catboost_rank,catboost_rank,boosted_tree_catboost,ranking,,152.340026,456.096534,-303.756509,25.051426,0.460706,250.233569,78.495782
66,quantbench_supervised_alstm_mse_macro_budget_d...,alstm_mse,sequence_attention_recurrent,mse,macro_budget_defensive_v1,150.236020,456.096534,-305.860515,16.226762,0.405456,-2.954465,25.786730
40,quantbench_supervised_extra_trees_rank,extra_trees_rank,tree_ensemble,ranking,,149.918656,456.096534,-306.177879,24.677793,0.449035,325.009533,78.137709
65,quantbench_supervised_alstm_mse,alstm_mse,sequence_attention_recurrent,mse,,148.184986,456.096534,-307.911548,25.429727,0.429026,337.854882,70.637141
145,quantbench_supervised_rolling_sequence_lag_rid...,rolling_sequence_lag_ridge_mse,custom,custom,,142.734271,456.096534,-313.362264,15.253971,0.429026,369.321446,69.595691
25,quantbench_supervised_xgboost_mse,xgboost_mse,boosted_tree_xgboost,mse,,131.169368,456.096534,-324.927166,26.361735,0.455704,654.293118,71.210966
30,quantbench_supervised_lightgbm_mse,lightgbm_mse,boosted_tree_lightgbm,mse,,129.001647,456.096534,-327.094887,26.829847,0.439030,618.549066,72.134718


,profile,model_name,paper_family,objective,risk_overlay,latest_return,latest_buy_hold_return,latest_relative_return,latest_max_dd,latest_avg_pos,proxy_return,proxy_max_dd
20,quantbench_ensemble_hazard_rf_logistic_v1,None,None,None,,35.239517,456.096534,-420.857018,17.113472,0.213884,192.180511,47.870092
4,quantbench_ensemble_guarded_rf_logistic_v1,None,None,None,,29.810211,456.096534,-426.286323,17.504261,0.287186,250.816294,47.145779
0,quantbench_ensemble_lowdd_mlp_v1,None,None,None,,28.170484,456.096534,-427.926050,17.938469,0.229954,43.170834,45.967561
3,quantbench_ensemble_defensive_logistic_v1,None,None,None,,27.250312,456.096534,-428.846222,17.423478,0.238089,72.794366,44.705274
1,quantbench_ensemble_lowdd_ridge_v1,None,None,None,,26.966724,456.096534,-429.129811,16.881901,0.219396,35.258738,44.044557
2,quantbench_ensemble_extra_logistic_v1,None,None,None,,25.491781,456.096534,-430.604753,18.564641,0.225755,28.607388,45.802637
7,quantbench_ensemble_confirmed_reentry_rf_logis...,None,None,None,,24.723087,456.096534,-431.373448,15.142460,0.195619,77.168071,34.554181
22,quantbench_ensemble_external_crash_v19v22_v1,None,None,None,,24.443089,456.096534,-431.653446,10.260835,0.137844,166.842013,17.512531
21,quantbench_ensemble_external_crash_condvol_v1,None,None,None,,22.431288,456.096534,-433.665246,10.738702,0.142071,120.426340,17.902942
5,quantbench_ensemble_guarded_hist_logistic_v1,None,None,None,,20.975509,456.096534,-435.121026,17.504261,0.265110,247.457440,47.336439


,profile,source_path,eligible,score,failed_checks,proxy_return,proxy_max_dd,stress_relative_return,stress_max_dd,stress_avg_pos,stress_window_avg_pos,stress_regime_avg_pos,bull_return,bull_buy_and_hold_return,bull_capture
0,quantbench_ensemble_sequence_fragile_panic_bud...,generated:quantbench_ensemble_sequence_fragile...,True,390.506821,[],316.520167,19.742257,78.029331,14.365664,0.100494,0.086678,0.100494,143.688894,191.773509,0.749264
1,quantbench_ensemble_sequence_fragile_preshock_v1,generated:quantbench_ensemble_sequence_fragile...,False,384.100448,['max_drawdown'],333.061242,20.549701,79.068428,13.997291,0.100988,0.084343,0.100988,142.237607,191.773509,0.741696
2,quantbench_ensemble_sequence_tapered_reentry_v1,generated:quantbench_ensemble_sequence_tapered...,False,381.172610,['max_drawdown'],328.660811,20.549621,79.071045,13.996299,0.100988,0.084341,0.100988,145.056021,191.773509,0.756392
3,quantbench_ensemble_sequence_full_rebase_tail_v1,generated:quantbench_ensemble_sequence_full_re...,False,366.314774,['max_drawdown'],323.538434,24.011303,79.076552,13.994212,0.100988,0.084336,0.100988,152.929783,191.773509,0.797450
4,quantbench_ensemble_sequence_panic_chip_floor_v1,generated:quantbench_ensemble_sequence_panic_c...,False,286.520953,"['max_drawdown', 'stress_avg_pos']",265.421463,20.495085,78.426753,14.285860,0.099506,0.083704,0.099506,123.987470,191.773509,0.646531
5,quantbench_ensemble_market_graph_panic_chip_fl...,generated:quantbench_ensemble_market_graph_pan...,False,279.370021,"['max_drawdown', 'stress_avg_pos']",259.848123,20.528030,78.426781,14.285835,0.099506,0.083704,0.099506,121.214738,191.773509,0.632072
6,quantbench_ensemble_lstm_panic_chip_floor_v1,generated:quantbench_ensemble_lstm_panic_chip_...,False,262.722586,"['max_drawdown', 'stress_avg_pos']",249.300435,21.145315,78.426390,14.286197,0.099506,0.083705,0.099506,114.252943,191.773509,0.595770
7,quantbench_ensemble_tcn_panic_chip_floor_v1,generated:quantbench_ensemble_tcn_panic_chip_f...,False,243.920806,"['max_drawdown', 'stress_avg_pos']",232.770022,21.143833,78.427679,14.285000,0.099506,0.083703,0.099506,109.883226,191.773509,0.572984
8,quantbench_ensemble_external_crash_v19v22_v1,generated:quantbench_ensemble_external_crash_v...,False,149.830463,"['stress_avg_pos', 'bull_capture']",166.842013,17.512531,76.164217,12.382956,0.063911,0.067905,0.063911,32.360549,191.773509,0.168744
9,quantbench_ensemble_guarded_rf_logistic_v1,generated:quantbench_ensemble_guarded_rf_logis...,False,111.035327,"['max_drawdown', 'stress_avg_pos', 'stress_max...",250.816294,47.145779,54.166601,32.877233,0.080000,0.163948,0.080000,104.776506,191.773509,0.546355


,family,paper_family,current_soxl_surface,coverage,next_step
0,tree_xgboost_lightgbm_catboost,XGBoost / LightGBM / CatBoost tree models,"xgboost_mse, lightgbm_mse, and catboost_rank a...",implemented,promote only if the expanded model matrix pass...
1,sequence_lstm_gru_alstm_sfm_darnn_hawkes,LSTM / GRU / ALSTM / SFM / DA-RNN / Hawkes-GRU...,"lstm_mse, gru_mse, alstm_mse, sfm_mse, darnn_m...",implemented,compare recurrent and attention recurrent sign...
2,sequence_mixer_transformer,TCN / MLP-Mixer / DLinear / PatchTST / Autofor...,"tcn_mse, mlp_mixer_mse, dlinear_mse, patch_tst...",implemented,keep only sequence-family signals that improve...
3,relational_gnn_hypergraph,GCN / GAT / RGCN / RSR / Hypergraph relational...,"market_graph_mse, gcn_mse, gat_mse, rgcn_mse, ...",implemented,promote graph-derived signals only if they add...
4,objectives_clf_ic_mse_ranking,CLF / IC / MSE / Ranking,common supervised target builder and CLF/IC/MS...,implemented,use objective metrics and risk-gate failures t...
5,utility_rl_execution,utility maximization / RL / execution,"PPO reward, risk budget, execution overlays exist",implemented,use QuantBench gate for promotion selection
6,rolling_continual,3/6/12 month rolling updates,rolling supervised refit paths and rolling haz...,implemented,keep cadence only if QuantBench gate improves;...
7,seed_ensemble,seed ensemble for overfit mitigation,low-correlation signal blend paths and macro p...,implemented,keep only blends that improve the QuantBench g...


## Persist Run Summary

In [9]:
best_supervised = supervised_latest_df.iloc[0].to_dict() if not supervised_latest_df.empty else {}
best_ensemble = ensemble_latest_df.iloc[0].to_dict() if not ensemble_latest_df.empty else {}
best_supervised_ytd_2026 = supervised_ytd_2026_df.iloc[0].to_dict() if not supervised_ytd_2026_df.empty else {}
best_ensemble_ytd_2026 = ensemble_ytd_2026_df.iloc[0].to_dict() if not ensemble_ytd_2026_df.empty else {}
best_gate = ensemble_screen_df.iloc[0].to_dict() if not ensemble_screen_df.empty else {}
summary = {
    "experiment": "quantbench_paper_soxl_v2",
    "paper": {
        "title": "QuantBench: Benchmarking AI Methods for Quantitative Investment",
        "arxiv": "https://arxiv.org/abs/2504.18600",
    },
    "data_refresh_status": data_refresh_status,
    "data": {
        "feature_rows": int(len(features)),
        "feature_count": int(features.shape[1]),
        "feature_end": feature_end,
        "close_end": close_end,
        "model_test_end": model_test_end,
        "windows": windows,
    },
    "architecture": {
        "implemented_model_count": int(len(expanded_model_names)),
        "model_names": list(expanded_model_names),
        "coverage_rows": coverage_df.to_dict("records"),
    },
    "supervised": {
        "decision": supervised_report["decision"],
        "gate_eligible_count": int(supervised_screen_df["eligible"].sum()) if not supervised_screen_df.empty else 0,
        "latest_best_row": best_supervised,
        "ytd_2026_best_row": best_supervised_ytd_2026,
    },
    "ensemble": {
        "decision": ensemble_report["decision"],
        "gate_eligible_count": int(ensemble_screen_df["eligible"].sum()) if not ensemble_screen_df.empty else 0,
        "latest_best_row": best_ensemble,
        "ytd_2026_best_row": best_ensemble_ytd_2026,
        "gate_best_row": best_gate,
    },
    "artifacts": {
        "notebook": str((CARD / "quantbench_paper_soxl_v2.ipynb").relative_to(ROOT)),
        "data_refresh_status_json": str((CARD / "data_refresh_status.json").relative_to(ROOT)),
        "supervised_model_specs_csv": str((CARD / "supervised_model_specs.csv").relative_to(ROOT)),
        "supervised_candidate_screen_csv": str((CARD / "supervised_candidate_screen.csv").relative_to(ROOT)),
        "ensemble_candidate_screen_csv": str((CARD / "ensemble_candidate_screen.csv").relative_to(ROOT)),
        "ensemble_payloads_json": str((CARD / "ensemble_payloads.json").relative_to(ROOT)),
        "summary_json": str((CARD / "summary.json").relative_to(ROOT)),
    },
    "decision": "expanded_architecture_backtest_recorded",
    "next_action": "Use the best eligible ensemble row for follow-up walk-forward replay; do not promote any single supervised model without passing the 20% MaxDD gate.",
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True, allow_nan=True) + "\n", encoding="utf-8")
print(json.dumps({
    "decision": summary["decision"],
    "model_test_end": model_test_end,
    "implemented_model_count": summary["architecture"]["implemented_model_count"],
    "supervised_best_latest": summary["supervised"]["latest_best_row"].get("profile"),
    "ensemble_best_latest": summary["ensemble"]["latest_best_row"].get("profile"),
    "supervised_best_ytd_2026": summary["supervised"]["ytd_2026_best_row"].get("profile"),
    "ensemble_best_ytd_2026": summary["ensemble"]["ytd_2026_best_row"].get("profile"),
    "ensemble_gate_best": summary["ensemble"]["gate_best_row"].get("profile"),
}, indent=2, sort_keys=True))

{
  "decision": "expanded_architecture_backtest_recorded",
  "ensemble_best_latest": "quantbench_ensemble_defensive_logistic_v1",
  "ensemble_best_ytd_2026": "quantbench_ensemble_hazard_rf_logistic_v1",
  "ensemble_gate_best": "quantbench_ensemble_sequence_fragile_panic_budget_v1",
  "implemented_model_count": 28,
  "model_test_end": "2026-06-04",
  "supervised_best_latest": "quantbench_supervised_tcn_mse",
  "supervised_best_ytd_2026": "quantbench_supervised_tcn_mse"
}


## Conclusion

This run implements and executes the full QuantBench-style SOXL model matrix in `src`: 28 supervised specs spanning linear/CLF/IC/MSE/ranking, XGBoost/LightGBM/CatBoost-style boosted trees, LSTM/GRU/ALSTM/SFM/DA-RNN/Hawkes-GRU, TCN/MLP-Mixer/DLinear/PatchTST/Autoformer-lite, and MarketGraph/GCN/GAT/RGCN/RSR/Hypergraph relational baselines. The coverage ledger now marks these model/objective families as implemented, with native GBDT packages kept explicit opt-in because the local native OpenMP stack can conflict with Torch in one process.

Data freshness: the executable backtest used the canonical local daily SOXL/SOXX/VIX panel through 2026-06-04. Live refresh probes in this run found Yahoo API returning 429 for SOXL/SOXX/^VIX, CBOE VIX available through 2026-06-05, and Futu OpenD not reachable, so the notebook did not hand-stitch a 2026-06-05 SOXL/SOXX OHLCV row from mixed web sources. Intraday daily-safe features are present through 2026-06-05, but full-history model training remains on the canonical daily panel.

Performance: no single supervised model passed the strict SOXL QuantBench gate. The highest latest-window single model was `quantbench_supervised_tcn_mse`: 2025-2026 return 492.65% versus SOXL buy-and-hold 849.40%, MaxDD 46.76%; 2026 YTD return 219.75% versus buy-and-hold 456.10%, MaxDD 13.29%. It is not promotable because full proxy MaxDD is 71.86%.

The path-ensemble screen has one eligible row: `quantbench_ensemble_sequence_fragile_panic_budget_v1`. Full proxy return is 316.52% with MaxDD 19.74%, 2022 stress relative defense 78.03%, stress MaxDD 14.37%, macro-stress average position 10.05%, and 2023-2024 bull capture 74.93%. On the latest 2025-2026 slice it returns 46.05% with MaxDD 12.30%; on 2026 YTD it returns 17.25% with MaxDD 12.30%. This remains the best gate-compliant model, but it materially under-captures the 2025-2026 SOXL rally, so the next branch should improve bull-regime release without weakening the panic-budget drawdown cap.